### Ситников Сергей Олегович М3239
# **Лабораторная 1**

## Задача 1 (Вариант 1)
В городе с населением $n+1$ человек некто узнает новость. Он передает ее первому встречному, тот еще одному и т.д. На каждом шаге услышавший новость может сообщить её любому из $n$ человек с одинаковыми вероятностями. Найти вероятность того, что в течение $r$ единиц времени новость никто не услышит дважды. Решить ту же задачу в предположении, что на каждом шаге услышавший на предыдущей итерации сообщение передает его группе из $m$ человек.

В первой версии на $j$-том шаге у нас $j$ человек осведомлены о новости, остаётся $n+1-j$ людей, которые неосведомлены. Чтобы на $r$-ом шаге никто не узнал новость дважды, эта новость должна быть произнесена за всё время только тем, кто о ней ещё не был осведомлен. Итого формула выглядит как 
$$ P(r) = \frac{n}{n} * \frac{n - 1}{n} * \frac{n - 2}{n} * \cdots * \frac{n - r + 1}{n} = \prod_{k=1}^{r} \frac{n-k+1}{n}=\frac{n!}{(n-r)!n^r},\quad 0 \le r \le n$$
При $r>n$ вероятность равна нулю.

In [1]:
import random
import math
random.seed("Sitnikov Sergey")

def simulate_one(n: int, r: int) -> bool:
    knows = [False] * (n + 1)
    knows[0] = True
    current_teller = 0
    for _ in range(r):
        index_chosen = random.randint(0, n - 1)
        if index_chosen >= current_teller:
            index_chosen += 1
        if knows[index_chosen]:
            return False
        knows[index_chosen] = True
        current_teller = index_chosen
    return True

COUNT_EXPERIMENTS = 200_000
print("n     r      simulation    formula       delta to formula")
for n in range(4, 10):
    for r in range(1, n + 1):
        good_count = 0
        for _ in range(COUNT_EXPERIMENTS):
            good_count += simulate_one(n, r)
        simulation = good_count / COUNT_EXPERIMENTS
        formula = math.factorial(n) / math.factorial(n - r) / (n**r)
        delta = (simulation - formula) / formula * 100
        print(f"{n:<3} {r:>3} {simulation:>13.6f} {formula:>13.6f} {delta:+10.2f}%")

n     r      simulation    formula       delta to formula
4     1      1.000000      1.000000      +0.00%
4     2      0.751525      0.750000      +0.20%
4     3      0.374110      0.375000      -0.24%
4     4      0.094495      0.093750      +0.79%
5     1      1.000000      1.000000      +0.00%
5     2      0.800570      0.800000      +0.07%
5     3      0.481250      0.480000      +0.26%
5     4      0.191185      0.192000      -0.42%
5     5      0.038530      0.038400      +0.34%
6     1      1.000000      1.000000      +0.00%
6     2      0.832680      0.833333      -0.08%
6     3      0.556110      0.555556      +0.10%
6     4      0.277125      0.277778      -0.24%
6     5      0.092230      0.092593      -0.39%
6     6      0.015110      0.015432      -2.09%
7     1      1.000000      1.000000      +0.00%
7     2      0.856665      0.857143      -0.06%
7     3      0.612035      0.612245      -0.03%
7     4      0.351735      0.349854      +0.54%
7     5      0.149465      0.1

Теперь вторая версия задачи, теперь сообщается новость не одному человеку за раз, а сразу m людям. Тогда на $j$-ом шагу нужно выбрать m людей только среди тех, кто еще не осведомлён. Неосведомлённых людей $n - (j-1)m$ (на каждом шагу осведомляется на m людей больше, это всё при учёте, что мы не хотим, чтобы одному и тому же человеку дважды была рассказана новость). Передатчик на $j$-ом шаге выбирается среди тех, кому рассказали новость на $(j-1)$-ом шаге. Значит формула будет выглядеть так:
$$ P(r) = \prod_{j=1}^r \frac{C_{n-(j-1)m}^m}{C_n^m}, \quad mr \le n$$
При $mr > n$ вероятность равна нулю.

In [43]:
import random
import math
random.seed("Sitnikov Sergey")

def simulate_one(n: int, m: int, r: int) -> bool:
    knows = [False] * (n + 1)
    knows[0] = True
    current_teller = 0
    for _ in range(r):
        possible_receivers = [
            person for person in range(n + 1)
            if person != current_teller
        ]
        group = random.sample(possible_receivers, m)
        for person in group:
            if knows[person]:
                return False
            knows[person] = True
        current_teller = random.choice(group)
    return True


COUNT_EXPERIMENTS = 200_000
print("n     m   r      simulation    formula       delta to formula")

for n in range(4, 10):
    for m in range(1, n + 1):
        for r in range(1, n // m + 1):
            good_count = 0
            for _ in range(COUNT_EXPERIMENTS):
                good_count += simulate_one(n, m, r)
            simulation = good_count / COUNT_EXPERIMENTS
            formula = math.prod(
                math.comb(n - (j - 1) * m, m) / math.comb(n, m)
                for j in range(1, r + 1)
            )
            delta = (simulation - formula) / formula * 100
            print(
                f"{n:<3} {m:>3} {r:>3} "
                f"{simulation:>13.6f} {formula:>13.6f} "
                f"{delta:+10.2f}%"
            )

n     m   r      simulation    formula       delta to formula
4     1   1      1.000000      1.000000      +0.00%
4     1   2      0.750490      0.750000      +0.07%
4     1   3      0.375810      0.375000      +0.22%
4     1   4      0.093080      0.093750      -0.71%
4     2   1      1.000000      1.000000      +0.00%
4     2   2      0.166465      0.166667      -0.12%
4     3   1      1.000000      1.000000      +0.00%
4     4   1      1.000000      1.000000      +0.00%
5     1   1      1.000000      1.000000      +0.00%
5     1   2      0.799920      0.800000      -0.01%
5     1   3      0.479905      0.480000      -0.02%
5     1   4      0.191495      0.192000      -0.26%
5     1   5      0.038300      0.038400      -0.26%
5     2   1      1.000000      1.000000      +0.00%
5     2   2      0.297990      0.300000      -0.67%
5     3   1      1.000000      1.000000      +0.00%
5     4   1      1.000000      1.000000      +0.00%
5     5   1      1.000000      1.000000      +0.00%
6 